# 🟢 LangSmith — Tier 1: Multi-turn conversation

**This is the hero scenario for LangSmith.** Threads view is LangSmith's strongest UX advantage over Langfuse and Galileo for chat use cases.

Three turns sharing `thread_id="tier1-multi-turn-langsmith"`. After running, the LangSmith Threads tab will render them as a chat conversation.

**What to look for in the LangSmith UI**:
- Project → **Threads** tab (top nav inside the project)
- Click `tier1-multi-turn-langsmith` → see a chat-style render of the 3 turns
- Each turn expands into its full Run tree on hover/click
- Cost + latency aggregated per thread (top of the panel)

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "observability-comparison")

from langchain_core.messages import HumanMessage
from shared.workflow import build_agent, MULTI_TURN_CONVERSATION

agent = build_agent(prompt_source="langsmith")

SESSION_ID = "tier1-multi-turn-langsmith"
print(f"thread_id: {SESSION_ID}")
for i, turn in enumerate(MULTI_TURN_CONVERSATION, 1):
    print(f"  Turn {i}: {turn}")

In [ ]:
messages = []
for turn_idx, user_turn in enumerate(MULTI_TURN_CONVERSATION, 1):
    messages.append(HumanMessage(content=user_turn))
    config = {
        "run_name": f"turn_{turn_idx}",
        "tags": ["tier1", "multi_turn", "langsmith", f"turn_{turn_idx}"],
        "metadata": {
            "thread_id": SESSION_ID,
            "session_id": SESSION_ID,
            "turn": turn_idx,
        },
    }
    out = agent.invoke({"messages": messages}, config=config)
    messages = out["messages"]

    final = messages[-1].content
    if isinstance(final, list):
        final = " ".join(p.get("text", "") for p in final if isinstance(p, dict))
    print(f"\n--- Turn {turn_idx} ---")
    print(f"User : {user_turn}")
    print(f"Agent: {final[:300]}")

print(f"\nDone. Open https://smith.langchain.com -> {os.environ['LANGSMITH_PROJECT']} -> Threads -> {SESSION_ID}.")

## Deck takeaway

**LangSmith Threads** is the strongest chat-debug UX of the three platforms. The Threads tab renders the same multi-turn data as a chat interface — exactly what you want when debugging a conversational agent.

Same data in **Langfuse Sessions** = a list of traces. Same data in **Galileo Sessions** = a list with metric badges per trace. Both useful, but visually less aligned with how product/PM stakeholders think about chatbots.